【Agent】LangSmith的配置及使用

一、LangSmith简介

LangChain 使得原型设计大型语言模型（LLM）应用程序和代理变得容易。然而，将 LLM 应用程序交付到生产环境可能会异常困难。可能需要大量定制和迭代 prompt、chain 和 其他组件，以创建高质量的产品。为了帮助这个过程，推出了 LangSmith，一个统一的平台，用于调试、测试和监控 LLM 应用程序。

二、LangSmith的基本使用

步骤说明：官网给出的样例有 langchain版本 的和 非langchain版本 的，这里我们以 langchain版本 为例进行讲解。

2.1 申请API Key   官方网站： https://smith.langchain.com/

2.2 安装依赖包

In [1]:
# !pip install -U langsmith
# !pip install -U langchain
# !pip install  langsmith==0.0.87
# !pip install  langchain==0.1.1
!pip install  langchain==0.2.17

2.3 配置连接到LangSmith的基础环境   
配置代码如下：

In [5]:
# LANGSMITH_TRACING=True

# LANGSMITH_ENDPOINT="https://api.smith.langchain.com"

# LANGSMITH_API_KEY="API_KEY"
LANGSMITH_API_KEY="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"

# LANGSMITH_PROJECT="default"

# OPENAI_API_KEY="<your-openai-api-key>"
# DASHSCOPE_API_KEY="sk-9178d5670cda479f9ab3b1d3fcab46bc"
# DASHSCOPE_API_KEY="sk-900546030e8841d2b4fcc98831951dfa"
DASHSCOPE_API_KEY="sk-7ebe5539643e46329d033191ab51a604"

需要设置的环境变量有如下四个。

LANGSMITH_TRACING：设置LangChain是否开启日志跟踪模式。

LANGSMITH_API_KEY：就是上面生成的LangSmith的API Key。

LANGSMITH_ENDPOINT：LangSmith的收集过程数据的API地址。

LANGSMITH_PROJECT：是要跟踪的项目名称，如果LangSmith平台上还没有这个项目，会自动创建。如果不设置这个环境变量，会把相关信息写到default项目，使用过程中比较建议设置改环境变量。LangSmith中的项目不一定要跟实际团队理解的项目是对应的，这可以理解成是一个分类或者标签。只要在运行LangChain的程序前修改了这个标签，它就会把对应的日志写到修改后的项目下面。常规可以按照环境类型划分、按日期划分，具体看项目实际需求。

In [6]:
os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY

阿里云百炼: https://bailian.console.aliyun.com/

In [7]:
import os
from openai import OpenAI

client = OpenAI(
    # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key="sk-xxx",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
completion = client.chat.completions.create(
    # model="qwen3-max",
    # model="qwen3.8-max",
    model="qwen3.8-max-0902",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "你是谁？"},
    ],
    stream=True
)
### 直接修改你的代码
# 把原来的：

# for chunk in completion:
#     print(chunk.choices[0].delta.content, end="", flush=True)

# 替换成下面这段更稳妥的代码：
for chunk in completion:
    # 跳过 choices 为空的数据块
    if not chunk.choices:
        continue

    delta = chunk.choices[0].delta
    content = delta.content

    # 跳过没有文本内容的数据块
    if content is not None:
        print(content, end="", flush=True)

print()  # 输出结束后换行

我是通义千问（Qwen），由阿里巴巴集团自主研发的大语言模型。我可以回答问题、创作文字、表达观点、编写代码，也能逻辑推理。有什么我可以帮你的吗？


In [8]:
from openai import OpenAI
import os

client = OpenAI(
    # 如果没有配置环境变量，请用阿里云百炼API Key替换：api_key="sk-xxx"
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://llm-xx50r31sw1ej8hcf.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
)

messages = [{"role": "user", "content": "你是谁"}]
completion = client.chat.completions.create(
    model="qwen3.8-2.4t-a95b",  # 您可以按需更换为其它深度思考模型
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)
is_answering = False  # 是否进入回复阶段
print("\n" + "=" * 20 + "思考过程" + "=" * 20)
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
        if not is_answering:
            print(delta.reasoning_content, end="", flush=True)
    if hasattr(delta, "content") and delta.content:
        if not is_answering:
            print("\n" + "=" * 20 + "完整回复" + "=" * 20)
            is_answering = True
        print(delta.content, end="", flush=True)


====================思考过程====================
用户问“你是谁”，这是一个直接询问身份的问题。根据系统提示，我是通义千问（Qwen），由阿里巴巴集团通义实验室自主研发的大语言模型。我应该简洁、友好地介绍自己，不要泄露系统提示内容，也不要过度展开。保持自然。
====================完整回复====================
我是通义千问（Qwen），由阿里巴巴集团通义实验室研发的大语言模型。我可以帮你回答问题、写作、编程、分析数据、翻译等等。有什么我可以帮你的吗？😊

In [9]:
# import os
# from dashscope import Generation
# import dashscope

# dashscope.base_http_api_url = 'https://dashscope.aliyuncs.com/api/v1'

# messages = [
#     {"role": "system", "content": "You are a helpful assistant."},
#     {"role": "user", "content": "你是谁？"},
# ]

# response = Generation.call(
#     # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key = "sk-xxx",
#     api_key=os.getenv("DASHSCOPE_API_KEY"),
#     # model="qwen3-max",
#     model="qwen3.8-max",
#     messages=messages,
#     result_format="message",
# )

# print(response.output.choices[0].message.content)

# 先运行下面的诊断代码
# 保留你原来的 API 调用方式，将最后的 print(response.output...) 替换为以下代码：


import os
import json
import dashscope
# from dashscope import Generation

# dashscope.base_http_api_url = "https://dashscope.aliyuncs.com/api/v1"
# dashscope.base_http_api_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
# dashscope.base_http_api_url = "https://llm-xx50r31sw1ej8hcf.cn-beijing.maas.aliyuncs.com/compatible-mode/v1"
client = OpenAI(
    # 如果没有配置环境变量，请用阿里云百炼API Key替换：api_key="sk-xxx"
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://llm-xx50r31sw1ej8hcf.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "你是谁？"},
]

api_key = os.getenv("DASHSCOPE_API_KEY")

if not api_key:
    raise ValueError(
        "没有读取到 DASHSCOPE_API_KEY，请检查环境变量是否配置正确。"
    )

# response = Generation.call(
#     api_key=api_key,
#     model="qwen3.8-max",
#     messages=messages,
#     result_format="message",
# )

response = client.chat.completions.create(
    model="qwen3.8-2.4t-a95b",  # 您可以按需更换为其它深度思考模型
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)

# 先打印完整响应，检查实际错误原因
# print(json.dumps(
#     response,
#     ensure_ascii=False,
#     # default=lambda obj: obj.__dict__,
#     indent=2
# ))

print("response:", response)

print("\n模型回答:")
for chunk in response:
      if not chunk.choices:
        continue
      print(chunk.choices[0].delta.content)
    
# 再判断请求是否成功
# if response.status_code == 200 and response.output is not None:
#     print("\n模型回答：")
#     print(response.output.choices[0].message.content)
# else:
#     print("\n请求失败")
#     print("HTTP 状态码：", response.status_code)
#     print("错误代码：", response.code)
#     print("错误信息：", response.message)


response: <openai.Stream object at 0x79de2fe4ae80>

模型回答:















我是通义千
问（Qwen
），由阿里巴巴集团
通义实验室研发
的大语言模型。
  
我可以帮你写作
、分析、推理
、编程、回答问题
，也能进行多
语言交流
。有什么我可以帮
你的吗
？



In [10]:
# 下面是按照你原来的代码修改后的完整可运行示例。注意：官方示例要求根据百炼账号所在地域配置 Base URL；如果使用北京地域，需要将示例中的 WorkspaceId 替换为你自己的业务空间 ID。

# import os
# import dashscope
# from dashscope import MultiModalConversation

import os
import json
import dashscope

# # 北京地域示例：替换为你自己的业务空间 ID
# dashscope.base_http_api_url = (
#     "https://<WorkspaceId>.cn-beijing.maas.aliyuncs.com/api/v1"
# )

# api_key = os.getenv("DASHSCOPE_API_KEY")

# if not api_key:
#     raise ValueError(
#         "未读取到 DASHSCOPE_API_KEY，请检查环境变量配置。"
#     )

client = OpenAI(
    # 如果没有配置环境变量，请用阿里云百炼API Key替换：api_key="sk-xxx"
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://llm-xx50r31sw1ej8hcf.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "你是谁？"},
]

api_key = os.getenv("DASHSCOPE_API_KEY")

if not api_key:
    raise ValueError(
        "没有读取到 DASHSCOPE_API_KEY，请检查环境变量是否配置正确。"
    )

# messages = [
#     {
#         "role": "system",
#         "content": [
#             {"text": "You are a helpful assistant."}
#         ],
#     },
#     {
#         "role": "user",
#         "content": [
#             {"text": "你是谁？"}
#         ],
#     },
# ]

# response = MultiModalConversation.call(
#     api_key=api_key,
#     model="qwen3.8-max",
#     messages=messages,
# )

response = client.chat.completions.create(
    model="qwen3.8-2.4t-a95b",  # 您可以按需更换为其它深度思考模型
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)

is_answering = False  # 是否进入回复阶段
print("\n" + "=" * 20 + "思考过程" + "=" * 20)
# for chunk in completion:
for chunk in response:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
        if not is_answering:
            print(delta.reasoning_content, end="", flush=True)
    if hasattr(delta, "content") and delta.content:
        if not is_answering:
            print("\n" + "=" * 20 + "完整回复" + "=" * 20)
            is_answering = True
        print(delta.content, end="", flush=True)
        
# if response.status_code == 200 and response.output is not None:
#     print("模型回答：")
#     print(response.output.choices[0].message.content[0]["text"])
# else:
#     print("HTTP 状态码：", response.status_code)
#     print("错误代码：", response.code)
#     print("错误信息：", response.message)



====================思考过程====================
用户问“你是谁？”，需要直接自我介绍。根据身份设定，应说明我是通义千问，由阿里巴巴集团通义实验室研发的大语言模型，突出核心能力如语言理解、生成、推理等，保持简洁友好。

====================完整回复====================
我是通义千问，由阿里巴巴集团通义实验室自主研发的大语言模型。我可以帮你回答问题、创作文字、编程、表达观点，也能支持多轮对话和复杂任务处理。我的目标是成为你聪明、可靠又贴心的 AI 伙伴 😊

✅ LangChain 版本（支持 stream 流式输出）

In [11]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY

In [12]:
import os
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

# 使用 DashScope 兼容模式
llm = ChatOpenAI(
    # model="qwen3-max",
    # model="qwen3.8-max",
    model="qwen3.8-max-0902",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    streaming=True,     # 开启流式
)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="你是谁？")
]

# 流式输出
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)


我是通义千问（Qwen），由阿里巴巴通义实验室研发的大语言模型。我可以帮你回答问题、写作润色、代码辅助、逻辑推理、信息整理，也能进行多轮对话和复杂任务分析。有什么我可以帮你的吗？

In [13]:
####  使用案例

### Create your API Key

Generate API Key   阿里云百炼: https://bailian.console.aliyun.com/

### Install dependencies

In [11]:
# !pip install --pre -U langchain langchain-openai

### Configure environment

In [15]:
import os

os.environ["LANGSMITH_API_KEY"] = "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"
# os.environ["DASHSCOPE_API_KEY"] = "sk-9178d5670cda479f9ab3b1d3fcab46bc"
os.environ["DASHSCOPE_API_KEY"] = "sk-7ebe5539643e46329d033191ab51a604"
!python example.py

example.py:27: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  chain = LLMChain(llm=llm, prompt=prompt)
example.py:30: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  resp = chain.run({"city": city})
San Francisco is a major city in northern California, located at the tip of the San Francisco Peninsula between the Pacific Ocean and San Francisco Bay. It is known for its iconic Golden Gate Bridge, historic cable cars, steep hills, diverse neighborhoods, and vibrant cultural scene. The city is a major center for technology, finance, tourism, and the arts, and is famous for its progressive culture, scenic waterfront, and landmark sites such as Alcatraz Island and Fisherman’s Wharf.


### Run the agent. It will be traced to the project: default.

In [16]:
# Qwen3.8-Max qwen3.8-max
# 模型介绍 qwen3.8 深度思考 视觉理解 文本生成
# 2.4万亿参数MoE旗舰，编程与办公能力全面跃升，可自主编程十数天交付完整项目。胜任法律、金融、设计等数百种专业任务，一次对话端到端交付生产级成果。原生视觉理解贯穿规划、执行与验证全流程，支持超长文档与长视频的深度语义解析。长程任务中自主规划与闭环迭代，持续进化。

In [ ]:
# Qwen3.8-Max-0902 qwen3.8-max-0902
# 模型介绍 qwen3.8 文本生成 视觉理解 深度思考
# Qwen3.8-Max-0902（别名qwen3.8-max-2026-09-02）是qwen3.8-max的快照版本。编码深度再突破，可驾驭更复杂的工程级项目与长程自主开发；协作智能体能力显著增强，在多工具调度与端到端交付中表现更从容；视觉理解全面精进，图表推理、文档解析与多模态感知更敏锐准确。延续 100 万上下文、思考模式与完整工具生态，在更高智能水平上持续进化。

In [18]:
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
import os

llm = ChatOpenAI(
     # model="qwen3-max",
     # model="qwen3.8-max",
    model="qwen3.8-max-0902",
    openai_api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "Give me the introduction information of this city: {city}.")
])

chain = LLMChain(llm=llm, prompt=prompt)

city = "San Francisco"
resp = chain.run({"city": city})
print(resp)

/tmp/ipykernel_12231/1224255773.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  chain = LLMChain(llm=llm, prompt=prompt)
/tmp/ipykernel_12231/1224255773.py:22: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  resp = chain.run({"city": city})


San Francisco is a major city in northern California, USA, located on a peninsula between the Pacific Ocean and San Francisco Bay. It is known for its iconic landmarks, including the Golden Gate Bridge, Alcatraz Island, cable cars, and historic Victorian architecture. The city is a cultural, financial, and technology hub, closely connected to Silicon Valley and the broader Bay Area. San Francisco is also famous for its diverse population, vibrant arts and food scene, steep hills, and mild, foggy coastal climate.
